## 개요
블로그 리뷰 텍스트를 태그 사전과 대조해 `places_mapo.csv`의 `tags` 컬럼을 채움.

- **팝업(`source="blog_popup"`)**: `nolda-장소가공.ipynb`가 이미 채워둔 `review_text`(원본 블로그 본문)를 그대로 재사용 — 새 API 호출 없음
- **일반 장소(`source="local"`)**: 실데이터 검증 결과 노이즈/오탐 문제가 발견돼 보류 중 (자세한 내용은 `PLACE_DATA_PIPELINE.md` 4-2-1 참고)

설계 문서: [PLACE_DATA_PIPELINE.md](PLACE_DATA_PIPELINE.md) 4절

## 1. 공통 매칭 로직
`nolda_common.py`의 `TAG_KEYWORDS`/`NEGATION_FLIP`을 대조. 매칭된 키워드 바로 뒤 10자 이내에 부정 표현이 있으면
매칭을 버리거나(반대 태그가 없는 경우) 반대 태그로 재분류함. 리뷰(문서/문장) 단위로 언급 여부를 세서, 같은 단위 안에서
키워드가 여러 번 나와도 1건으로만 카운트 — threshold 이상 언급된 태그만 채택.

In [1]:
import re

import pandas as pd

from nolda_common import NEGATION_FLIP, TAG_KEYWORDS

NEGATION_PATTERN = re.compile(r"없|안\s|못|아니")
THRESHOLD = 2  # 실데이터로 튜닝 필요 (PLACE_DATA_PIPELINE.md 4-6 참고)


def to_review_units(text, min_len=8):
    """본문/리뷰 텍스트를 매칭 단위(줄)로 쪼갬. 줄바꿈이 없는 짧은 텍스트는 그 자체를 한 단위로 취급"""
    lines = [line.strip() for line in str(text).split("\n")]
    lines = [line for line in lines if len(line) >= min_len]
    return lines if lines else [str(text)]


def count_tag_mentions(review_units):
    """단위(문서/줄)별로 태그 언급 여부를 판정해서 {태그: 언급된 단위 수}로 집계"""
    counts = {}
    for text in review_units:
        matched = set()
        for tag, keywords in TAG_KEYWORDS.items():
            for kw in keywords:
                for m in re.finditer(re.escape(kw), text):
                    following = text[m.end():m.end() + 10]
                    if NEGATION_PATTERN.search(following):
                        flip = NEGATION_FLIP.get(tag)
                        if flip:
                            matched.add(flip)
                        # 반대 태그가 없으면 그냥 버림 (무효화)
                    else:
                        matched.add(tag)
        for tag in matched:
            counts[tag] = counts.get(tag, 0) + 1
    return counts


def resolve_tags(review_units, threshold=THRESHOLD):
    counts = count_tag_mentions(review_units)
    return sorted(tag for tag, c in counts.items() if c >= threshold)

## 2. 팝업(`blog_popup`) 태그 매칭
`review_text`를 줄 단위로 쪼개서 매칭 -> threshold 이상 언급된 태그만 채택.
`source="local"` 행은 이번엔 건드리지 않음(기존 `tags` 값 그대로 유지).

In [2]:
CSV_PATH = "data/places_mapo.csv"

df = pd.read_csv(CSV_PATH)

popup_mask = df["source"] == "blog_popup"
tag_counts_summary = {}
tagged_popup_count = 0

for idx in df[popup_mask].index:
    review_text = df.at[idx, "review_text"]
    if pd.isna(review_text):
        continue
    units = to_review_units(review_text)
    tags = resolve_tags(units)
    df.at[idx, "tags"] = str(tags)  # 리스트를 바로 넣으면 pandas가 다중값 할당으로 오해해서 에러남 -> 문자열로 저장
    if tags:
        tagged_popup_count += 1
    for tag in tags:
        tag_counts_summary[tag] = tag_counts_summary.get(tag, 0) + 1

df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

print(f"팝업 총 {popup_mask.sum()}건 중 태그가 채택된 곳: {tagged_popup_count}건")
print("태그별 채택 건수:", tag_counts_summary)
df[popup_mask][["name", "tags"]].head(20)

팝업 총 48건 중 태그가 채택된 곳: 10건
태그별 채택 건수: {'가족동반': 1, '데이트': 1, '사진맛집': 8, '웨이팅있음': 2}


,name,tags
1179,홍대 꾸감 더 현대 대구 팝업 스토어,[]
1180,매력 팝업,"['가족동반', '데이트', '사진맛집']"
1181,홍대 AK플라자 이누야샤 팝업 스토어,['사진맛집']
1182,홍대 AK플라자 이누야샤 팝업 스토어,[]
1183,UP STORE 라이브 청바지 팝업 스토어,[]
1184,이누야샤 팝업 @홍대,[]
1185,홍대 다마고치 팝업 스토어,[]
1186,어노브 파워퍼프걸 홍대 올리브영 팝업 스토어,['사진맛집']
1187,명탐정 코난: 하이웨이의 타천사 &amp; 야이바 팝업스토어,['사진맛집']
1188,명동 팝업 스토어,[]


## 3. 일반 장소(`local`) 태그 매칭 — 보류
실데이터 검증 결과 검색 기반 리뷰 수집이 노이즈/오탐 문제가 있어 보류 중.
재개 시 이 자리에 구현. 자세한 내용과 재개 옵션은 `PLACE_DATA_PIPELINE.md` 4-2-1 참고.